# 02 — Netflix Semantic Search Pipeline
### Cortex-RAG Series | github.com/ather-ops
---
**What you will build:**
- Full EDA on Netflix dataset with 5 professional visualizations
- Combine title + director + cast + genres + description into one text
- Generate semantic embeddings for every Netflix title
- Build a search function — ask anything, get relevant movies/shows
- Two modes: sklearn cosine similarity vs custom from scratch

> **This is the bridge between raw data and a full RAG system.**
> After this notebook, you add a vector database and an LLM — and you have RAG.

## Step 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported")
print("="*60)

## Step 2 — Load Data with Error Handling

Always wrap data loading in try/except.
If the file is missing or corrupted, we exit cleanly instead of crashing.

In [ ]:
try:
    df = pd.read_csv("Netflix_Dataset.csv")
    print("Data loaded successfully")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
    print(df.head(5))
except FileNotFoundError:
    print("File not found — check Netflix_Dataset.csv is in same folder")
    exit()
except Exception as e:
    print(f"Error: {e}")
    exit()

## Step 3 — EDA: Understand the Data

In [ ]:
print("="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

print("Basic statistics:")
print(df.describe())
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)

## Step 4 — Fill Missing Values

In [ ]:
# Fill missing values with Unknown
df["director"] = df["director"].fillna("Unknown")
df["cast"]     = df["cast"].fillna("Unknown")
df["country"]  = df["country"].fillna("Unknown")

print("Missing values after filling:")
print(df[["director", "cast", "country"]].isnull().sum())
print()
print("Sample after cleaning:")
print(df[["title", "director", "country"]].head())

## Step 5 — EDA Visualizations

5 professional charts covering:
1. Content type distribution (Movies vs TV Shows)
2. Top content-producing countries
3. Most popular genres
4. Content release growth over years
5. Audience rating distribution

In [ ]:
sns.set_theme(style="whitegrid")
primary    = "#f97316"
secondary  = "#06b6d4"
dark       = "#0f172a"
bg         = "#03040f"

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor(bg)
fig.suptitle('Netflix Content Analysis Dashboard',
             color='white', fontsize=16, fontweight='bold', y=1.01)

for ax in axes.flat:
    ax.set_facecolor(dark)
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#1e293b')

# Chart 1: Movies vs TV Shows
type_counts = df['type'].value_counts()
axes[0,0].pie(type_counts.values, labels=type_counts.index,
              autopct='%1.1f%%', colors=[primary, secondary],
              textprops={'color':'white'}, startangle=90)
axes[0,0].set_title('Movies vs TV Shows', color='white', fontweight='bold')

# Chart 2: Top Countries
top_countries = df[df['country'] != 'Unknown']['country'].value_counts().head(8)
bars = axes[0,1].barh(top_countries.index[::-1], top_countries.values[::-1],
                       color=primary, alpha=0.85)
axes[0,1].set_title('Top Content-Producing Countries', color='white', fontweight='bold')
axes[0,1].set_xlabel('Number of Titles', color='#94a3b8')
axes[0,1].tick_params(colors='#94a3b8')
for spine in axes[0,1].spines.values(): spine.set_color('#1e293b')
for bar, val in zip(bars, top_countries.values[::-1]):
    axes[0,1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                   str(val), va='center', color='white', fontsize=9)

# Chart 3: Top Genres
genres = df['listed_in'].str.split(', ').explode()
genre_counts = genres.value_counts().head(8)
axes[0,2].barh(genre_counts.index[::-1], genre_counts.values[::-1],
               color=secondary, alpha=0.85)
axes[0,2].set_title('Top 8 Most Popular Genres', color='white', fontweight='bold')
axes[0,2].set_xlabel('Count', color='#94a3b8')
axes[0,2].tick_params(colors='#94a3b8')
for spine in axes[0,2].spines.values(): spine.set_color('#1e293b')

# Chart 4: Release Year Trend
yearly = df.groupby('release_year')['show_id'].count().reset_index()
axes[1,0].plot(yearly['release_year'], yearly['show_id'],
               color=primary, linewidth=2.5, marker='o', markersize=4)
axes[1,0].fill_between(yearly['release_year'], yearly['show_id'],
                        color=primary, alpha=0.15)
axes[1,0].set_title('Content Release Growth Over Years', color='white', fontweight='bold')
axes[1,0].set_xlabel('Year', color='#94a3b8')
axes[1,0].set_ylabel('Titles Released', color='#94a3b8')
axes[1,0].tick_params(colors='#94a3b8')
for spine in axes[1,0].spines.values(): spine.set_color('#1e293b')
axes[1,0].grid(True, alpha=0.2)

# Chart 5: Rating Distribution
rating_counts = df['rating'].value_counts()
colors_rating = [primary if i == 0 else secondary if i == 1 else '#94a3b8'
                 for i in range(len(rating_counts))]
axes[1,1].bar(rating_counts.index, rating_counts.values,
              color=colors_rating, alpha=0.85)
axes[1,1].set_title('Audience Rating Distribution', color='white', fontweight='bold')
axes[1,1].set_xlabel('Rating', color='#94a3b8')
axes[1,1].set_ylabel('Count', color='#94a3b8')
axes[1,1].tick_params(colors='#94a3b8', axis='x', rotation=45)
for spine in axes[1,1].spines.values(): spine.set_color('#1e293b')

# Chart 6: Movies vs Shows by Year
movies = df[df['type']=='Movie'].groupby('release_year').size()
shows  = df[df['type']=='TV Show'].groupby('release_year').size()
axes[1,2].plot(movies.index, movies.values, color=primary,
               label='Movies', linewidth=2)
axes[1,2].plot(shows.index, shows.values, color=secondary,
               label='TV Shows', linewidth=2)
axes[1,2].set_title('Movies vs TV Shows Over Time', color='white', fontweight='bold')
axes[1,2].set_xlabel('Year', color='#94a3b8')
axes[1,2].tick_params(colors='#94a3b8')
for spine in axes[1,2].spines.values(): spine.set_color('#1e293b')
axes[1,2].legend(facecolor=dark, labelcolor='white')
axes[1,2].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('netflix_eda_dashboard.png', dpi=120, bbox_inches='tight', facecolor=bg)
plt.show()
print("EDA Dashboard saved as netflix_eda_dashboard.png")

## Step 6 — Combine Text for Embedding

We combine all relevant text fields into one string per title.

**Why?** The embedding model needs all context in one place.
Title alone is not enough — we need genre, description, cast, director.

In [ ]:
df["combined_text"] = (
    df["title"]       + " " +
    df["director"]    + " " +
    df["cast"]        + " " +
    df["listed_in"]   + " " +
    df["description"]
)

print("Combined text for first 3 titles:")
print("="*60)
for i in range(3):
    print(f"\n[{i}] {df['combined_text'].iloc[i][:200]}...")

## Step 7 — Generate Embeddings

We use `all-MiniLM-L6-v2` — a lightweight but powerful model.

Each Netflix title becomes a 384-dimensional vector capturing its full meaning.

In [ ]:
print("Loading SentenceTransformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings for all titles...")
embeddings = model.encode(df["combined_text"].tolist(), show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Each of {embeddings.shape[0]} titles = vector of {embeddings.shape[1]} numbers")
print(f"Total numbers stored: {embeddings.shape[0] * embeddings.shape[1]:,}")

## Step 8 — Custom Cosine Similarity vs Sklearn

In [ ]:
# Custom from scratch — for understanding
def cosine_scratch(v1, v2):
    dot   = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    return dot / (norm1 * norm2)

# Verify both give same result
sample1 = embeddings[0]
sample2 = embeddings[1]

scratch_score = cosine_scratch(sample1, sample2)
sklearn_score = cosine_similarity([sample1], [sample2])[0][0]

print(f"Custom cosine similarity : {scratch_score:.6f}")
print(f"Sklearn cosine similarity: {sklearn_score:.6f}")
print(f"Difference               : {abs(scratch_score - sklearn_score):.10f}")
print("Both give identical results — sklearn just runs faster on large data")

## Step 9 — Semantic Search Function

The search function:
1. Encodes your query into a 384-dim vector
2. Computes similarity with all Netflix embeddings
3. Returns top-K most similar titles

This is **semantic search** — it understands meaning, not just keywords.

In [ ]:
def search(query, top_k=3, use_sklearn=True):
    """
    Search Netflix titles by semantic meaning.
    
    Args:
        query    : Natural language search query
        top_k    : Number of results to return
        use_sklearn: True = sklearn (fast), False = custom from scratch
    
    Returns:
        DataFrame with top-K matching titles
    """
    query_emb = model.encode([query])

    if use_sklearn:
        scores = cosine_similarity(query_emb, embeddings).flatten()
    else:
        scores = np.array([cosine_scratch(query_emb[0], emb) for emb in embeddings])

    top_indices = np.argsort(scores)[::-1][:top_k]
    results = df.iloc[top_indices][["title", "director", "description"]].copy()
    results["score"] = scores[top_indices].round(4)
    return results


print("Search function ready")
print("Usage: search('your query here', top_k=3)")

## Step 10 — Test All Queries

In [ ]:
queries = [
    "What is Netflix",
    "Tell me about romantic movies",
    "Tell me about comedy movies",
    "Tell me about action movies",
    "Movies directed by Steven Spielberg",
    "Movies starring Tom Hanks",
    "Crime thriller TV shows",
    "Indian movies and shows",
    "Documentary about nature",
    "Shows for children and family"
]

for q in queries:
    print(f"\nQuery: {q}")
    print("-"*55)
    results = search(q, top_k=3)
    for _, row in results.iterrows():
        print(f"  Score: {row['score']} | {row['title']} | Dir: {row['director']}")
        print(f"  {row['description'][:100]}...")
    print()

## Step 11 — Visualize Search Scores

In [ ]:
# Visualize scores for one query
query = "romantic love story movie"
query_emb = model.encode([query])
scores = cosine_similarity(query_emb, embeddings).flatten()
top_10 = np.argsort(scores)[::-1][:10]

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#03040f')
ax.set_facecolor('#0f172a')

titles  = df.iloc[top_10]['title'].values
sc      = scores[top_10]
colors  = ['#f97316' if s > 0.6 else '#06b6d4' if s > 0.4 else '#94a3b8' for s in sc]

bars = ax.barh(range(len(titles)), sc, color=colors, alpha=0.85)
ax.set_yticks(range(len(titles)))
ax.set_yticklabels(titles, color='white', fontsize=9)
ax.set_xlabel('Cosine Similarity Score', color='#94a3b8')
ax.set_title(f'Top 10 Results for: "{query}"', color='white', fontsize=12, fontweight='bold')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values(): spine.set_color('#1e293b')
ax.axvline(x=0.5, color='#f97316', linestyle='--', alpha=0.5, label='0.5 threshold')
ax.legend(facecolor='#0f172a', labelcolor='white')

for bar, val in zip(bars, sc):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('search_scores.png', dpi=120, bbox_inches='tight', facecolor='#03040f')
plt.show()

## Step 12 — Save Results to New CSV

In [ ]:
# Save cleaned data + embedding dimensions to new CSV
print("Saving results to My_New_Netflix_Dataset.csv...")

# Save sample query results for all queries
all_results = []
for q in queries[:5]:
    res = search(q, top_k=3)
    res['query'] = q
    all_results.append(res)

results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv('My_New_Netflix_Dataset.csv', index=False)

print(f"Saved {len(results_df)} rows")
print(f"Columns: {results_df.columns.tolist()}")
print()
print(results_df.head())

## What You Built — Summary

| Step | What happened |
|------|--------------|
| Load data | Error-handled CSV load |
| EDA | 6-panel professional dashboard |
| Clean data | Missing values filled |
| Combine text | title + director + cast + genres + description |
| Embeddings | 384-dim vector per Netflix title |
| Cosine similarity | Custom from scratch + sklearn — identical results |
| Semantic search | Query → meaning → top-K results |
| Visualize | Score bar chart for search results |
| Save | New CSV with query results |

---

## What Comes Next in Cortex-RAG

```
Notebook 03 — Vector Databases (Chroma)
  Store embeddings permanently
  Query by similarity at scale
  Add metadata filters

Notebook 04 — LLM Response Generation
  Connect HuggingFace or OpenAI
  Generate natural language answers

Notebook 05 — Complete RAG System
  User query → Retrieve → Generate → Answer
  This is the full production RAG pipeline
```

Built by Ather Assadullah — github.com/ather-ops/Cortex-RAG